In [1]:
from pathlib import Path
import torch
import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer

torch.set_num_threads(2)
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if p.name == "miniproject2" and (p / "data/train.csv").is_file()), None)
if root is None:
    root = Path.cwd() / "miniproject2"
if not (root / "data/train.csv").is_file():
    raise FileNotFoundError("miniproject2 또는 note 폴더에서 실행하세요.")
# 기존 캐시가 있으면 재사용. 없으면 새 폴더에 모델을 다운로드합니다.
shared_cache = root.parent / "Note/miniproject1/.model_cache"
cache_dir = shared_cache if shared_cache.is_dir() else root / "note/.model_cache"
local_only = shared_cache.is_dir()
from transformers import AutoModelForMaskedLM
model_id = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=str(cache_dir), local_files_only=local_only)
model = AutoModelForMaskedLM.from_pretrained(model_id, cache_dir=str(cache_dir), local_files_only=local_only).eval()
print("모델:", model_id)
print("클래스:", type(model).__name__)


모델: klue/bert-base
클래스: BertForMaskedLM


In [2]:
display(pd.DataFrame([{"구조": "Transformer encoder", "층 수": model.config.num_hidden_layers,
                       "hidden 차원": model.config.hidden_size, "attention head": model.config.num_attention_heads,
                       "어휘 수": model.config.vocab_size}]))
print("MLM 출력 head:")
print(model.cls)


,구조,층 수,hidden 차원,attention head,어휘 수
0,Transformer encoder,12,768,12,32000


MLM 출력 head:
BertOnlyMLMHead(
  (predictions): BertLMPredictionHead(
    (transform): BertPredictionHeadTransform(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (transform_act_fn): GELUActivation()
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    )
    (decoder): Linear(in_features=768, out_features=32000, bias=True)
  )
)


In [3]:
text = "비가 와서 [MASK]을 챙겻다."
inputs = tokenizer(text, return_tensors="pt")
ids = inputs["input_ids"][0].tolist()
display(pd.DataFrame({"위치": range(len(ids)), "토큰": tokenizer.convert_ids_to_tokens(ids), "토큰 ID": ids}))
mask_positions = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().flatten()
assert mask_positions.numel() == 1
mask_index = mask_positions.item()
print("입력:", text)
print("마스크 위치:", mask_index)


,위치,토큰,토큰 ID
0,0,[CLS],2
1,1,비,1187
2,2,##가,2116
3,3,와서,5241
4,4,[MASK],4
5,5,을,1498
6,6,[UNK],1
7,7,.,18
8,8,[SEP],3


입력: 비가 와서 [MASK]을 챙겻다.
마스크 위치: 4


In [4]:
with torch.inference_mode():
    logits = model(**inputs).logits
    probabilities = logits[0, mask_index].softmax(dim=-1)
    scores, candidate_ids = probabilities.topk(5)
print("logits shape (배치, 길이, 어휘):", tuple(logits.shape))
candidates = []
for score, token_id in zip(scores.tolist(), candidate_ids.tolist()):
    word = tokenizer.decode([token_id]).strip()
    candidates.append({"후보 토큰": tokenizer.convert_ids_to_tokens(token_id), "확률": score,
                       "복원 문장": text.replace("[MASK]", word, 1)})
display(pd.DataFrame(candidates))


logits shape (배치, 길이, 어휘): (1, 9, 32000)


,후보 토큰,확률,복원 문장
0,우산,0.368020,비가 와서 우산을 챙겻다.
1,창문,0.048007,비가 와서 창문을 챙겻다.
2,바람,0.042138,비가 와서 바람을 챙겻다.
3,눈,0.033937,비가 와서 눈을 챙겻다.
4,길,0.028844,비가 와서 길을 챙겻다.


In [5]:
comparisons = []
for sentence in ["비가 와서 [MASK]을 챙겼다.", "비가 와서 [MASK]을 챙겻다."]:
    batch = tokenizer(sentence, return_tensors="pt")
    position = (batch["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
    with torch.inference_mode():
        probs = model(**batch).logits[0, position].softmax(dim=-1)
    scores, ids = probs.topk(5)
    comparisons.append({"문맥": sentence, "1위 후보": tokenizer.decode([ids[0].item()]),
                        "1위 확률": scores[0].item(),
                        "Top-5": ", ".join(tokenizer.decode([i]) for i in ids.tolist())})
display(pd.DataFrame(comparisons))


,문맥,1위 후보,1위 확률,Top-5
0,비가 와서 [MASK]을 챙겼다.,우산,0.841228,"우산, 가방, 도시락, 수건, 이불"
1,비가 와서 [MASK]을 챙겻다.,우산,0.368020,"우산, 창문, 바람, 눈, 길"
